In [3]:
# Importing modules
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
#import seaborn
import os
#import os as sns
import nltk
nltk.download('words') # Downloads a list of English words, which can be used for various tasks like checking if a given string is a valid English word.
nltk.download('maxent_ne_chunker') #Downloads the Maximum Entropy Named Entity Chunker, which is a model used for named entity recognition (NER). This tool identifies named entities like people, organizations, and locations from text.
nltk.download('punkt') #Downloads the Punkt sentence tokenizer, which is used to split text into sentences and further tokenize sentences into words. 
nltk.download('averaged_perceptron_tagger') #Downloads the Averaged Perceptron Tagger, which is used for part-of-speech (POS) tagging
import re #provides support for regular expressions in Python. Regular expressions are used for searching, matching, and manipulating strings based on specific patterns
from nltk import pos_tag #Imports the `pos_tag` function from the Natural Language Toolkit (NLTK), which assigns part-of-speech tags to each word in a text. It classifies words into categories like noun, verb, adjective, etc., which is useful for syntactic analysis.
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize #Imports the `word_tokenize` function from NLTK, which splits a text into individual words (or tokens).
from collections import Counter #Imports the `Counter` class from the `collections` module. `Counter` is a dict subclass that is used to count hashable objects. It is particularly useful for counting frequencies of words or tokens in a text.
import gensim #a powerful library for topic modeling
from gensim.utils import simple_preprocess #a convenient method for preprocessing text. It tokenizes text by performing lowercasing, removing punctuation, and optionally removing short tokens or stemming words.
import gensim.corpora as corpora # These are used to create and manage the mappings between words and their integer ids in text analysis.
from gensim.corpora import Dictionary #Imports the `Dictionary` class from Gensim's `corpora` module, which is used to map words from text documents to unique integer ids.
nltk.download('stopwords')
import spacy #open-source library for advanced natural language processing (NLP) tasks for pre-processing
#from wordcloud import WordCloud, STOPWORDS
#import matplotlib.pyplot

from nltk.corpus import stopwords

nltk.download('stopwords')
sw = set(stopwords.words("english"))
#sw = STOPWORDS
os.chdir('..') #The `chdir('..')` function changes the current working directory to the parent directory.
# Read data into ReviewsOFD
#verbal_reports = pd.read_csv(working_dataset, encoding = 'utf-8')
df = pd.read_csv('K:\All_Data_Factors_Extraction.csv', encoding = 'utf-8')
#df = pd.read_csv('K:\Processed_Data_Output_v4.csv')
#ReviewsOFD = ReviewsOFD.Reviews.values.tolist()
#ReviewsOFD = [t.split(',') for t in ReviewsOFD]
# Print head
df.head()

[nltk_data] Downloading package words to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bbb4464\AppData\Roaming\nltk_data...
[nltk_data]   Package stopword

,ID,OBSR ID,Step Number,Keyword,Verbal Report,Original Statement,Reasoning,Grade
0,1,sec_0,1,Customer Expectations,The individual had expectations about the deli...,An estimated delivery time of max 35 min ended...,The individual had a clear expectation about t...,-100.0
1,1,sec_0,2,Perceived Quality,The individual experienced many issues with th...,It was great when it worked until it didn't. ....,The individual had a negative experience with ...,-100.0
2,1,sec_0,3,Perceived Value,The individual did not receive their order and...,One of the worst experiences with a helpdesk I...,The individual did not receive the value they ...,-100.0
3,1,sec_0,4,Perceived Image,The individual had a negative experience with ...,One of the worst experiences with a helpdesk I...,The individual's negative experience with the ...,-100.0
4,1,sec_0,5,Customer Satisfaction,The individual was extremely dissatisfied with...,One of the worst experiences with a helpdesk I...,The individual's experience was extremely nega...,-100.0


In [5]:
# Define the order of keywords or use unique keywords
keywords_order = [
    'Customer Expectations',
    'Perceived Quality',
    'Perceived Value',
    'Customer Satisfaction',
    'Customer Complaints',
    'Customer Loyalty',
    'Perceived Image',
    'Customer Trust',
]

# File path for the output Excel file
output_file = r'K:\All_Data_Factors_Extraction.xlsx'

# Use ExcelWriter to write each DataFrame to a corresponding sheet
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    for keyword in keywords_order:
        # Filter the data for each keyword
        keyword_data = df[df['Keyword'] == keyword][['ID', 'OBSR ID', 'Step Number', 'Verbal Report', 'Original Statement', 'Reasoning', 'Grade'
]]

        # Write the filtered data to a sheet named after the keyword
        keyword_data.to_excel(writer, sheet_name=keyword, index=False)

print(f"Data has been saved to {output_file}") 

Data has been saved to K:\All_Data_Factors_Extraction.xlsx


In [25]:
# Ensure necessary NLTK datasets are downloaded
#nltk.download('stopwords')

# File path for your CSV
file_path = r'K:\Customer Loyalty.csv'
df = pd.read_csv(file_path)

# Handle missing values by filling them with empty strings
df['Original Statement'] = df['Original Statement'].fillna('')

# Remove punctuation and convert text to lowercase
df['Original Statement'] = df['Original Statement'].apply(lambda x: re.sub('[,\.!?]', '', str(x).lower()))

# Print out the first few rows of the processed column for quick verification
print(df['Original Statement'].head())

# Assuming 'Verbal_Report_corrected' is another column containing textual data.
data = df['Original Statement']

# Build bigrams and trigrams capture word patterns and common expressions
bigram = gensim.models.Phrases(data, min_count=20, threshold=100)
trigram = gensim.models.Phrases(bigram[data], threshold=100)
bigram_mod = gensim.models.phrases.Phraser(bigram)
trigram_mod = gensim.models.phrases.Phraser(trigram)

# Use only the tagger in Spacy for faster processing A spaCy NLP model is loaded (`en_core_web_sm`) with the parser and named entity recognizer disabled. This speeds up processing when POS tagging and lemmatization are primarily required
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

# Define stopwords using NLTK
stop_words = set(stopwords.words('english'))
#The text is lemmatized using spaCy, which reduces words to their base form while retaining specific parts of speech tags (`NOUN`, `ADJ`, `VERB`, `ADV`).
def process_words(texts, stop_words=stop_words, allowed_tags=['NOUN', 'ADJ', 'VERB', 'ADV']): #A custom function `process_words()` is defined to process the text
    """Convert a document into a list of lowercase tokens, build bigrams-trigrams, implement lemmatization."""
    # Remove stopwords, short tokens, and letter accents
    texts = [[word for word in simple_preprocess(str(doc), deacc=True, min_len=3) if word not in stop_words] for doc in texts]

    texts_out = []
    
    # Implement lemmatization and filter out unwanted parts of speech tags, Another round of stopword removal and punctuation removal is conducted post-lemmatization
    for sent in texts:
        doc = nlp(" ".join(sent))
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_tags])
    
    # Remove stopwords and short tokens again after lemmatization
    texts_out = [[word for word in simple_preprocess(str(doc), deacc=True, min_len=3) if word not in stop_words] for doc in texts_out]
    
    return texts_out

# Process your text data
data_ready = process_words(data)

0    last time me and my wife have used uber eats a...
1          i just started using this app for a few day
2                             very upset with this app
3    there are a lot of issues with the app some of...
4                    worst food delivery app ever used
Name: Original Statement, dtype: object


In [26]:
def preprocess_and_extract_nouns(doc):
    # POS tagging to extract nouns
    tagged_words = pos_tag(doc) #Use `pos_tag` from NLTK to perform part-of-speech tagging on the document
    
    # Extract nouns (NN, NNS, NNP, NNPS are common noun tags)
    nouns = [word for word, pos in tagged_words if pos in ['NN', 'NNS', 'NNP', 'NNPS']] # Identify and extract words tagged as nouns, which include:
    #- `'NN'`: Noun, singular or mass - `'NNS'`: Noun, plural - `'NNP'`: Proper noun, singular - `'NNPS'`: Proper noun, plural
     # Return the list of extracted nouns.
    return nouns

# Since data_ready already contains tokenized documents,
# just apply the noun extraction directly.
#The noun extraction function is applied to each document in `data_ready`, storing the results in a new column `'Nouns'` in the `df` DataFrame.
# `data_ready` is presumed to contain lists of tokenized and processed words from the input documents.
df['Nouns'] = [preprocess_and_extract_nouns(doc) for doc in data_ready]

# Flatten the list of nouns and count frequency
#Flattening: Combine the lists of nouns into a single list (`all_nouns`).
all_nouns = sum(df['Nouns'].tolist(), [])  # Flatten list of lists
noun_freq = Counter(all_nouns) #counting: Use Python's `Counter` from the `collections` module to count the frequency of each noun

# Calculate total number of nouns for percentage calculations
total_nouns = sum(noun_freq.values())

# Prepare results with frequency and percentage
noun_stats = [(noun, freq, (freq / total_nouns) * 100) for noun, freq in noun_freq.most_common()]

# Convert results to a DataFrame for better display
noun_freq_df = pd.DataFrame(noun_stats, columns=['Noun', 'Frequency', 'Percentage'])

# Display the top nouns by frequency
print(noun_freq_df.sort_values(by='Frequency', ascending=False).head(100))

# Display the top nouns by frequency
top_nouns_df = noun_freq_df.sort_values(by='Frequency', ascending=False).head(100)
print(top_nouns_df)

# Save the result to an Excel file
output_file_path = 'K:/noun_frequency_v1.xlsx'  # Specify your desired file path and name
top_nouns_df.to_excel(output_file_path, index=False, sheet_name='TopNouns')

print(f"Top nouns by frequency have been saved to {output_file_path}")

           Noun  Frequency  Percentage
0         order       3493    5.510507
1           use       3481    5.491576
2           app       3397    5.359058
3       service       2164    3.413895
4          food       1880    2.965861
..          ...        ...         ...
95  convenience        109    0.171957
96      mention        107    0.168802
97   membership        107    0.168802
98        start        105    0.165646
99        worth        105    0.165646

[100 rows x 3 columns]
           Noun  Frequency  Percentage
0         order       3493    5.510507
1           use       3481    5.491576
2           app       3397    5.359058
3       service       2164    3.413895
4          food       1880    2.965861
..          ...        ...         ...
95  convenience        109    0.171957
96      mention        107    0.168802
97   membership        107    0.168802
98        start        105    0.165646
99        worth        105    0.165646

[100 rows x 3 columns]
Top nouns by fre